   
# Delta Table Maintenance Lab

Welcome to the Delta Table Maintenance lab! In this hands-on lab, you'll learn how to maintain and optimize Delta Lake tables for better performance and storage efficiency.

---

## Learning Objectives

By the end of this lab, you will be able to:

1. Understand why small files are a performance problem
2. Use **OPTIMIZE** to compact small files
3. Use **ZORDER** to co-locate related data
4. Use **VACUUM** to remove old files and save storage
5. Monitor table health with DESCRIBE DETAIL
6. Check table history with DESCRIBE HISTORY
7. Understand retention periods and data lifecycle
8. Use **Liquid Clustering** as the modern replacement for ZORDER
9. Use **ANALYZE TABLE** to collect statistics for the query optimizer

---

### **OPTIMIZE**
* **Problem:** Many small files slow down queries
* **Solution:** Combine small files into larger, optimized files
* **Benefit:** Faster queries, better compression
* **Optional:** ZORDER for data clustering

### **VACUUM**
* **Problem:** Old files accumulate and waste storage
* **Solution:** Delete files no longer needed
* **Benefit:** Reduced storage costs
* **Caution:** Affects time travel capability

### **Liquid Clustering**
* **Problem:** ZORDER requires manual re-optimization and doesn't support incremental clustering
* **Solution:** `CLUSTER BY` automatically organizes data on write and during OPTIMIZE
* **Benefit:** Simpler management, incremental clustering, changeable keys without full rewrite

### **ANALYZE TABLE**
* **Problem:** Query optimizer lacks statistics to choose efficient plans
* **Solution:** Compute table and column-level statistics
* **Benefit:** Better join strategies, more accurate cost estimates

---

## Lab Structure

This lab has **11 tasks** to complete:

1. Create a table with small files (simulate the problem)
2. Check table details (see the small files issue)
3. Run OPTIMIZE to compact files
4. Verify optimization results
5. Add more data and use ZORDER
6. Run VACUUM to clean up old files (DRY RUN)
7. Run VACUUM to actually delete files
8. Check table history
9. Final table health check
10. Liquid Clustering (the modern alternative to ZORDER)
11. ANALYZE TABLE (collect optimizer statistics)

**Each task includes:**
* Clear instructions
* Hints to guide you
* Solutions at the end (try first!)

---

## Dataset

We'll create our own table with booking data to simulate real-world scenarios.

---

**Let's get started!** 

## Task 1: Create a Table with Small Files

**The Problem:**

When data is written in many small batches, Delta tables end up with many small files. This hurts query performance!

**Your Challenge:**

Create a Delta table with booking data by writing multiple small batches.

**Requirements:**

1. Create a table called `workspace.default.bookings_lab`
2. Write data in **5 separate batches** (simulating incremental writes)
3. Each batch should have 1000 rows
4. Use `.coalesce(1)` to ensure each batch creates a small file
5. Use mode `append` for batches 2-5

**Data structure:**
* `booking_id` - INT (sequential)
* `customer_id` - INT (random 1-500)
* `booking_date` - DATE (random dates in 2024)
* `amount` - DOUBLE (random 50-1000)
* `region` - STRING (random: 'North', 'South', 'East', 'West')

---

**Write your code in the cell below:**

In [0]:
# TODO: Generate and write 5 batches of booking data
# Each batch: 1000 rows, coalesce(1) to create small files
# Batch 1: mode("overwrite"), Batches 2-5: mode("append")


### Hints for Task 1
<br/>
<details>
<summary><b>Hint 1:</b> Generating sample data</summary>

Use Python to generate data:
```python
import random
from datetime import datetime, timedelta

data = [
    (i, random.randint(1, 500), 
     (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
     round(random.uniform(50, 1000), 2),
     random.choice(['North', 'South', 'East', 'West']))
    for i in range(start_id, end_id)
]
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Creating DataFrame</summary>

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("booking_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("booking_date", StringType()),
    StructField("amount", DoubleType()),
    StructField("region", StringType())
])

df = spark.createDataFrame(data, schema)
```
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Writing batches</summary>

```python
# Batch 1 (overwrite)
df1.coalesce(1).write.mode("overwrite").saveAsTable("workspace.default.bookings_lab")

# Batch 2-5 (append)
df2.coalesce(1).write.mode("append").saveAsTable("workspace.default.bookings_lab")
```
</details>

## Task 2: Check Table Details

**Your Challenge:**

Inspect the table to see the small files problem.

**Requirements:**

1. Use `DESCRIBE DETAIL` to view table metadata
2. Look for:
   * `numFiles` - Should be around 5 (one per batch)
   * `sizeInBytes` - Total table size
   * `location` - Where the table is stored
3. Calculate the average file size

**Questions to answer:**
* How many files does the table have?
* What's the average file size?
* Is this optimal? (Hint: Ideal file size is 128MB-1GB)

---

**Write your code in the cell below:**

In [0]:
%sql
-- TODO: Use DESCRIBE DETAIL to inspect the table
-- Look at numFiles and sizeInBytes


### Hints for Task 2
<br/>
<details>
<summary><b>Hint 1:</b> DESCRIBE DETAIL syntax</summary>

```sql
DESCRIBE DETAIL catalog.schema.table_name
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Key columns to look at</summary>

Important columns:
* `numFiles` - Number of data files
* `sizeInBytes` - Total size in bytes
* `location` - Table location
* `format` - Should be 'delta'
</details>

## Task 3: Run OPTIMIZE

**The Problem:**

Your table has 5 small files. Reading many small files is slow because:
* More file open/close operations
* Less efficient compression
* More metadata to track
* Slower query performance

**Your Challenge:**

Run OPTIMIZE to compact the small files into larger, optimized files.

**Requirements:**

1. Use the `OPTIMIZE` command on your table
2. Run it using SQL (`%sql`)
3. Observe the output metrics:
   * `numFilesAdded` - New optimized files created
   * `numFilesRemoved` - Old small files marked for removal

**Syntax:**
```sql
OPTIMIZE catalog.schema.table_name
```

---

**Write your code in the cell below:**

In [0]:
%sql
-- TODO: Run OPTIMIZE on workspace.default.bookings_lab


### Hints for Task 3
<br/>
<details>
<summary><b>Hint 1:</b> OPTIMIZE syntax</summary>

```sql
OPTIMIZE workspace.default.bookings_lab
```

That's it! Just one line.
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Understanding the output</summary>

OPTIMIZE returns metrics:
* `numFilesAdded` - New optimized files (usually 1)
* `numFilesRemoved` - Old files marked for deletion (should be 5)
* `totalFilesSkipped` - Files already optimal
* `totalTimeMs` - Time taken
</details>
<br/>
<details>
<summary><b>Hint 3:</b> What happens</summary>

OPTIMIZE:
1. Reads all small files
2. Combines them into larger files
3. Writes new optimized files
4. Marks old files for deletion (but doesn't delete yet!)
5. Updates transaction log
</details>

## Task 4: Verify Optimization Results

**Your Challenge:**

Check if OPTIMIZE worked by inspecting the table again.

**Requirements:**

1. Run `DESCRIBE DETAIL` again on your table
2. Compare with Task 2 results:
   * `numFiles` - Should be fewer (ideally 1)
   * `sizeInBytes` - Should be similar (same data)
3. Calculate the new average file size

**Expected results:**
* Before OPTIMIZE: ~5 small files
* After OPTIMIZE: 1 larger file
* File size: Much larger per file

---

**Write your code in the cell below:**

In [0]:
%sql
-- TODO: Run DESCRIBE DETAIL again to see the changes
-- Compare numFiles before and after


### Hints for Task 4
<br/>
<details>
<summary><b>Hint 1:</b> Same command as Task 2</summary>

```sql
DESCRIBE DETAIL workspace.default.bookings_lab
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> What to look for</summary>

Compare:
* **Before:** numFiles = 5, small avg file size
* **After:** numFiles = 1, larger avg file size
* **Data:** sizeInBytes should be similar (same data, better compressed)
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Why files are better</summary>

Larger files mean:
* Fewer file operations
* Better compression
* Faster queries
* More efficient storage
</details>

## Task 5: Add More Data and Use ZORDER

**What is ZORDER?**

ZORDER BY co-locates related data in the same files, making queries on those columns much faster.

**Your Challenge:**

Add more data and optimize with ZORDER.

**Requirements:**

**Part A: Add more data**
1. Generate 2000 more rows (booking_id 5001-7000)
2. Append to the table
3. Use `.coalesce(2)` to create 2 more small files

**Part B: OPTIMIZE with ZORDER**
1. Run OPTIMIZE with ZORDER BY on the `region` column
2. This will cluster data by region for faster region-based queries

**Syntax:**
```sql
OPTIMIZE table_name ZORDER BY (column_name)
```

---

**Write your code in the cells below:**

In [0]:
# TODO: Generate 2000 more rows (booking_id 5001-7000)
# Append to the table with coalesce(2)


In [0]:
%sql
-- TODO: Run OPTIMIZE with ZORDER BY (region)


### Hints for Task 5
<br/>
<details>
<summary><b>Hint 1:</b> Generating more data</summary>

```python
# Similar to Task 1, but different ID range
data_batch6 = [
    (i, random.randint(1, 500), 
     (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
     round(random.uniform(50, 1000), 2),
     random.choice(['North', 'South', 'East', 'West']))
    for i in range(5001, 7001)
]
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> ZORDER syntax</summary>

```sql
OPTIMIZE workspace.default.bookings_lab
ZORDER BY (region)
```

You can ZORDER by multiple columns:
```sql
ZORDER BY (region, booking_date)
```
</details>
<br/>
<details>
<summary><b>Hint 3:</b> When to use ZORDER</summary>

Use ZORDER on columns that are:
* Frequently used in WHERE clauses
* Used for joins
* High cardinality (many distinct values)
* Not partition columns (use partitioning instead)

In our case: `region` is frequently filtered, so ZORDER helps!
</details>

## Task 6: Understand VACUUM

**The Problem:**

After OPTIMIZE, the old small files are still on disk! They're marked for deletion in the transaction log, but physically still exist.

**Why?**
* Time travel needs old files
* Concurrent readers might be using them
* Safety buffer before permanent deletion

**Your Challenge:**

Run VACUUM DRY RUN to see what would be deleted.

**Requirements:**

1. Use `VACUUM` with `DRY RUN` option
2. Set retention to 0 hours (for demo purposes only!)
3. Observe what files would be deleted

**Syntax:**
```sql
VACUUM table_name RETAIN 0 HOURS DRY RUN
```

**Important:** 
* DRY RUN shows what WOULD be deleted (doesn't actually delete)
* RETAIN 0 HOURS is only for demos - never use in production!
* Default retention is 7 days (168 hours)

---

**Write your code in the cell below:**

In [0]:
%sql
-- Alter retention duration (required for 0 hours in serverless workspaces)
ALTER TABLE workspace.default.bookings_lab
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

-- TODO: Run VACUUM with DRY RUN to preview deletions
-- Use RETAIN 0 HOURS for this demo


### Hints for Task 6
<br/>
<details>
<summary><b>Hint 1:</b> VACUUM DRY RUN syntax</summary>

```sql
VACUUM workspace.default.bookings_lab RETAIN 0 HOURS DRY RUN
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Disable retention check</summary>

For RETAIN 0 HOURS, you need to disable the safety check first:
```sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM workspace.default.bookings_lab RETAIN 0 HOURS DRY RUN
```
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Understanding output</summary>

DRY RUN shows:
* List of files that would be deleted
* These are the old small files from before OPTIMIZE
* No files are actually deleted (safe to run)
</details>

## Task 7: Run VACUUM (Actually Delete Files)

**Your Challenge:**

Now run VACUUM for real to delete the old files.

**Requirements:**

1. Disable the retention check (required for 0 hours)
2. Run VACUUM with RETAIN 0 HOURS (without DRY RUN)
3. Observe the output showing deleted files

**Commands needed:**
```sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM table_name RETAIN 0 HOURS
```

**Warning:** 
* This permanently deletes files!
* Time travel to old versions will fail after VACUUM
* In production, use 7+ days retention
* We use 0 hours only for demo purposes

---

**Write your code in the cell below:**

In [0]:
%sql
-- TODO: Disable retention check and run VACUUM
-- This will actually delete the old files


### Hints for Task 7
<br/>
<details>
<summary><b>Hint 1:</b> Two commands needed</summary>

```sql
-- Command 1: Disable safety check
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

-- Command 2: Run VACUUM
VACUUM workspace.default.bookings_lab RETAIN 0 HOURS
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> What happens</summary>

VACUUM:
1. Finds files older than retention period
2. Checks they're not in current table version
3. Permanently deletes them from storage
4. Frees up disk space
5. Returns list of deleted files
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Production retention</summary>

In production, use:
```sql
VACUUM table_name RETAIN 168 HOURS  -- 7 days (default)
VACUUM table_name RETAIN 720 HOURS  -- 30 days
```

Balance:
* Longer retention = More time travel, more storage cost
* Shorter retention = Less time travel, less storage cost
</details>

## Task 8: Check Table History

**Your Challenge:**

View the complete history of operations on your table.

**Requirements:**

1. Use `DESCRIBE HISTORY` to view all transactions
2. Look for these operations:
   * `CREATE OR REPLACE TABLE` (initial creation)
   * `WRITE` (your append operations)
   * `OPTIMIZE` (file compaction)
   * `VACUUM` (file deletion)
3. Examine the `operationMetrics` to see:
   * How many files were added/removed
   * How much data was processed

**Questions to answer:**
* How many versions does your table have?
* Which operations created the most files?
* What did OPTIMIZE do (check numFilesAdded/numFilesRemoved)?

---

**Write your code in the cell below:**

In [0]:
%sql
-- TODO: Use DESCRIBE HISTORY to view all operations


### Hints for Task 8
<br/>
<details>
<summary><b>Hint 1:</b> DESCRIBE HISTORY syntax</summary>

```sql
DESCRIBE HISTORY workspace.default.bookings_lab
```

Limit to recent operations:
```sql
DESCRIBE HISTORY workspace.default.bookings_lab LIMIT 10
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Key columns to examine</summary>

Important columns:
* `version` - Transaction number
* `timestamp` - When it happened
* `operation` - Type of operation
* `operationMetrics` - Detailed metrics (files, rows, bytes)
* `userName` - Who did it
</details>

## Task 9: Final Table Health Check

**Your Challenge:**

Perform a final health check on your optimized table.

**Requirements:**

1. Run `DESCRIBE DETAIL` one more time
2. Verify the table is in good shape:
   * `numFiles` - Should be 1 or very few
   * `sizeInBytes` - Total data size
   * Calculate average file size (should be larger now)
3. Query the table to ensure data is intact
4. Count total rows (should be 7000)

**Success criteria:**
* Fewer files than before
* Larger average file size
* All data still accessible
* No data loss

---

**Write your code in the cells below:**

In [0]:
%sql
-- TODO: Run DESCRIBE DETAIL to check final state


In [0]:
%sql
-- TODO: Query the table and count rows
-- Should have 7000 rows total


### Hints for Task 9
<br/>
<details>
<summary><b>Hint 1:</b> Check table details</summary>

```sql
DESCRIBE DETAIL workspace.default.bookings_lab
```
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Verify row count</summary>

```sql
SELECT COUNT(*) AS total_rows
FROM workspace.default.bookings_lab
```

Should be 7000 (5 batches of 1000 + 1 batch of 2000)
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Sample the data</summary>

```sql
SELECT * 
FROM workspace.default.bookings_lab
ORDER BY booking_id
LIMIT 20
```

Verify:
* booking_id ranges from 1 to 7000
* All columns present
* Data looks correct
</details>

## Task 10: Liquid Clustering – The Modern Alternative

**What is Liquid Clustering?**

Liquid clustering is the **recommended replacement** for both table partitioning and ZORDER. It uses the `CLUSTER BY` clause to automatically organize data for faster queries.

**Why is it better than ZORDER?**

| Feature | ZORDER | Liquid Clustering |
| --- | --- | --- |
| Incremental clustering | No (full rewrite) | Yes (only new data) |
| Change keys | Must re-OPTIMIZE all data | `ALTER TABLE` – no rewrite |
| Works on write | No | Yes (clustering on write) |
| Requires OPTIMIZE | Always | OPTIMIZE triggers reclustering |
| Compatible with partitioning | Yes | No (replaces partitioning) |

**Your Challenge:**

**Part A: Create a liquid-clustered table**
1. Create a new table `workspace.default.bookings_lc_lab` with `CLUSTER BY (region)`
2. Insert data from the existing `bookings_lab` table
3. Run `DESCRIBE DETAIL` to confirm clustering is configured

**Part B: Change clustering keys**
1. Use `ALTER TABLE` to change the clustering key to `(region, booking_date)`
2. Run `OPTIMIZE` to apply the new clustering
3. Try removing clustering with `ALTER TABLE ... CLUSTER BY NONE`

**Key syntax:**
```sql
-- Create with clustering
CREATE TABLE t CLUSTER BY (col) AS SELECT ...

-- Change keys (no rewrite!)
ALTER TABLE t CLUSTER BY (col1, col2)

-- Remove clustering
ALTER TABLE t CLUSTER BY NONE

-- Trigger reclustering
OPTIMIZE t
```

**Note:** Liquid clustering is **not compatible** with ZORDER. You cannot use `OPTIMIZE ... ZORDER BY` on a liquid-clustered table.

---

**Write your code in the cells below:**

In [0]:
%sql
-- TODO Part A: Create workspace.default.bookings_lc_lab with CLUSTER BY (region)
-- Insert data from workspace.default.bookings_lab
-- Then run DESCRIBE DETAIL to verify


In [0]:
%sql
-- TODO Part B: Change clustering keys and optimize
-- 1. ALTER TABLE to cluster by (region, booking_date)
-- 2. Run OPTIMIZE to apply new clustering
-- 3. (Optional) Remove clustering with CLUSTER BY NONE


### Hints for Task 10
<br/>
<details>
<summary><b>Hint 1:</b> Creating the clustered table</summary>

```sql
CREATE OR REPLACE TABLE workspace.default.bookings_lc_lab
CLUSTER BY (region)
AS SELECT * FROM workspace.default.bookings_lab;
```

Then check:
```sql
DESCRIBE DETAIL workspace.default.bookings_lc_lab
```

Look for the `clusteringColumns` field – it should show `["region"]`.
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Changing clustering keys</summary>

```sql
-- Change to multi-column clustering
ALTER TABLE workspace.default.bookings_lc_lab
CLUSTER BY (region, booking_date);

-- Trigger reclustering with new keys
OPTIMIZE workspace.default.bookings_lc_lab;
```

Note: Changing keys does NOT rewrite existing data. Only new writes and OPTIMIZE will use the new keys.
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Key differences from ZORDER</summary>

* ZORDER: `OPTIMIZE table ZORDER BY (col)` – full rewrite every time
* Liquid clustering: `ALTER TABLE table CLUSTER BY (col)` + `OPTIMIZE table` – incremental
* You **cannot** mix them: a liquid-clustered table rejects `ZORDER BY`
* Liquid clustering also enables **clustering on write** – new data is partially clustered automatically
* Use `DESCRIBE DETAIL` to see `clusteringColumns` on a liquid-clustered table
</details>

## Task 11: ANALYZE TABLE – Collect Optimizer Statistics

**Why do statistics matter?**

The query optimizer uses statistics to make smart decisions:
* Which join strategy to use (broadcast vs shuffle)
* Which table should be on the build side of a hash join
* How to order joins in multi-way joins
* How many rows to expect from filters

Without statistics, the optimizer guesses – and can choose slow plans!

**Your Challenge:**

**Part A: Compute table-level statistics**
1. Run `ANALYZE TABLE` on `workspace.default.bookings_lab` to compute row count and size
2. Use `DESC EXTENDED` to verify the statistics are stored

**Part B: Compute column-level statistics**
1. Run `ANALYZE TABLE ... FOR COLUMNS` on key columns (`region`, `amount`, `booking_date`)
2. Use `DESC EXTENDED` on a column to see min, max, distinct count, null count

**Part C: Compute Delta-specific statistics**
1. Run `ANALYZE TABLE ... COMPUTE DELTA STATISTICS` to compute file-level stats

**Key syntax:**
```sql
-- Table-level stats (row count + size)
ANALYZE TABLE t COMPUTE STATISTICS;

-- Table-level stats (size only, faster)
ANALYZE TABLE t COMPUTE STATISTICS NOSCAN;

-- Column-level stats
ANALYZE TABLE t COMPUTE STATISTICS FOR COLUMNS col1, col2;

-- Delta-specific file stats
ANALYZE TABLE t COMPUTE DELTA STATISTICS;
```

**Tip:** In production, enable **Predictive Optimization** on your Unity Catalog managed tables – it runs `ANALYZE TABLE` automatically!

---

**Write your code in the cells below:**

In [0]:
%sql
-- TODO Part A: Run ANALYZE TABLE to compute table-level statistics
-- Then use DESC EXTENDED to verify the stats are stored



In [0]:
%sql
-- TODO Part B: Run ANALYZE TABLE FOR COLUMNS on region, amount, booking_date
-- Then use DESC EXTENDED on a column to see detailed stats



### Hints for Task 11
<br/>
<details>
<summary><b>Hint 1:</b> Table-level statistics</summary>

```sql
-- Compute full stats (includes row count)
ANALYZE TABLE workspace.default.bookings_lab COMPUTE STATISTICS;

-- Verify with DESC EXTENDED – look for the "Statistics" row
DESC EXTENDED workspace.default.bookings_lab;
```

You should see something like: `Statistics: XXXX bytes, 7000 rows`
</details>
<br/>
<details>
<summary><b>Hint 2:</b> Column-level statistics</summary>

```sql
-- Compute column stats
ANALYZE TABLE workspace.default.bookings_lab
COMPUTE STATISTICS FOR COLUMNS region, amount, booking_date;

-- View stats for a specific column
DESC EXTENDED workspace.default.bookings_lab region;
```

Column stats include:
* `min` / `max` – value range
* `num_nulls` – null count
* `distinct_count` – cardinality
* `avg_col_len` / `max_col_len` – for string columns
</details>
<br/>
<details>
<summary><b>Hint 3:</b> Delta-specific statistics</summary>

```sql
-- Compute Delta file-level stats
ANALYZE TABLE workspace.default.bookings_lab COMPUTE DELTA STATISTICS;
```

This collects per-file min/max stats that enable **data skipping** – Delta can skip entire files that don't match your WHERE clause.

Combined with ZORDER or liquid clustering, this makes filtered queries very fast.
</details>

## Lab Complete!

Congratulations! You've successfully completed the OPTIMIZE and VACUUM lab.

### **What You Accomplished:**

* Created a table with small files (simulated the problem)  
* Used DESCRIBE DETAIL to diagnose issues  
* Ran OPTIMIZE to compact files  
* Used ZORDER to cluster data  
* Ran VACUUM DRY RUN to preview deletions  
* Ran VACUUM to free storage  
* Monitored table health with DESCRIBE HISTORY  
* Verified data integrity  
* Created a liquid-clustered table and changed clustering keys  
* Collected optimizer statistics with ANALYZE TABLE  

---

### **Key Takeaways:**

1. **Small files hurt performance** – Many small files slow down queries
2. **OPTIMIZE is your friend** – Compact files regularly
3. **ZORDER boosts queries** – Cluster by frequently filtered columns
4. **Liquid clustering is the future** – Prefer `CLUSTER BY` over `ZORDER` for new tables
5. **VACUUM saves money** – Delete old files to reduce storage costs
6. **ANALYZE TABLE helps the optimizer** – Collect stats for better query plans
7. **Balance retention** – Time travel vs storage costs
8. **Monitor regularly** – Use DESCRIBE DETAIL, DESCRIBE HISTORY, and DESC EXTENDED

---

### **Next Steps:**

* Apply OPTIMIZE/VACUUM to your production tables
* Migrate existing ZORDER tables to liquid clustering
* Set up scheduled jobs for workspacetenance
* Enable Predictive Optimization for automated stats and optimization
* Monitor query performance improvements
* Explore Delta Lake table properties

---

### **Resources:**

* [OPTIMIZE Documentation](https://docs.databricks.com/sql/language-manual/delta-optimize.html)
* [VACUUM Documentation](https://docs.databricks.com/sql/language-manual/delta-vacuum.html)
* [Liquid Clustering](https://docs.databricks.com/delta/clustering.html)
* [ANALYZE TABLE](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-aux-analyze-compute-statistics.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [Predictive Optimization](https://docs.databricks.com/optimizations/predictive-optimization.html)

# Complete Solutions

**Only look at these if you're stuck or want to verify your work!**

Try to solve the challenges yourself first. Learning happens through struggle and problem-solving!

---

## Solution: Task 1 (Create Table with Small Files)
<br/>
<details>

```python
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define schema
schema = StructType([
    StructField("booking_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("booking_date", StringType()),
    StructField("amount", DoubleType()),
    StructField("region", StringType())
])

# Generate and write 5 batches
for batch_num in range(1, 6):
    start_id = (batch_num - 1) * 1000 + 1
    end_id = batch_num * 1000 + 1
    
    # Generate data
    data = [
        (i, 
         random.randint(1, 500),
         (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
         round(random.uniform(50, 1000), 2),
         random.choice(['North', 'South', 'East', 'West']))
        for i in range(start_id, end_id)
    ]
    
    df = spark.createDataFrame(data, schema)
    
    # Write batch
    if batch_num == 1:
        df.coalesce(1).write.mode("overwrite").saveAsTable("workspace.default.bookings_lab")
    else:
        df.coalesce(1).write.mode("append").saveAsTable("workspace.default.bookings_lab")
    
    print(f"Batch {batch_num} written: {len(data)} rows")

print(f"Table created with 5000 rows in 5 small files")
```

**Key concepts:**
* Loop to create multiple batches
* `coalesce(1)` forces one file per batch
* First batch uses `overwrite`, rest use `append`
* This simulates real-world incremental writes

</details>

## Solution: Task 2 (Check Table Details)
<br/>
<details>

```sql
DESCRIBE DETAIL workspace.default.bookings_lab
```

**What you should see:**
* `numFiles`: 5
* `sizeInBytes`: ~500KB-1MB total

**Why this is bad:**
* Small files = many file operations
* Inefficient for Spark to process
* Slower queries
* More metadata overhead

</details>

## Solution: Task 3 (Run OPTIMIZE)
<br/>
<details>

```sql
OPTIMIZE workspace.default.bookings_lab
```

**Expected output:**
```
metrics:
  numFilesAdded: 1
  numFilesRemoved: 5
  totalFilesSkipped: 0
  totalTimeMs: ~1000-5000
```

**What happened:**
1. Delta Lake read all 5 small files
2. Combined them into 1 larger file
3. Wrote the new optimized file
4. Marked old files for deletion (in transaction log)
5. Old files still exist on disk (until VACUUM)

**Benefits:**
* Queries now read 1 file instead of 5
* Better compression
* Faster performance

</details>

## Solution: Task 4 (Verify Optimization)
<br/>
<details>

```sql
DESCRIBE DETAIL workspace.default.bookings_lab
```

**What changed:**

**Before OPTIMIZE:**
* numFiles: 5
* avg file size: ~100-200 KB

**After OPTIMIZE:**
* numFiles: 1
* avg file size: ~500KB-1MB

**Verification:**
```sql
SELECT COUNT(*) AS total_rows
FROM workspace.default.bookings_lab
```
Should still be 5000 rows (no data lost!)

**Key insight:** OPTIMIZE doesn't change data, just reorganizes files for better performance.

</details>

## Solution: Task 5 (Add Data and Use ZORDER)
<br/>
<details>

**Part A: Add more data**
```python
# Generate 2000 more rows
data_batch6 = [
    (i, 
     random.randint(1, 500),
     (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
     round(random.uniform(50, 1000), 2),
     random.choice(['North', 'South', 'East', 'West']))
    for i in range(5001, 7001)
]

df_batch6 = spark.createDataFrame(data_batch6, schema)
df_batch6.coalesce(2).write.mode("append").saveAsTable("workspace.default.bookings_lab")

print("Added 2000 more rows in 2 files")
```

**Part B: OPTIMIZE with ZORDER**
```sql
OPTIMIZE workspace.default.bookings_lab
ZORDER BY (region)
```

**What ZORDER does:**
* Co-locates data with same region values
* Makes queries filtering by region much faster
* Example: `WHERE region = 'North'` only reads relevant files

**When to use ZORDER:**
* Columns frequently used in WHERE clauses
* High cardinality columns
* Columns used for joins

</details>

## Solution: Task 6 (VACUUM DRY RUN)
<br/>
<details>

```sql
-- Alter retention duration (required for 0 hours in serverless workspaces)
ALTER TABLE workspace.default.bookings_lab
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

-- Run VACUUM DRY RUN
VACUUM workspace.default.bookings_lab RETAIN 0 HOURS DRY RUN
```

**Expected output:**
* List of file paths that would be deleted
* These are the old files from before OPTIMIZE
* No files are actually deleted (DRY RUN is safe)

**Why DRY RUN is useful:**
* Preview what will be deleted
* Verify retention period is correct
* Check if important files would be removed
* Safe to run in production

**Best practice:** Always run DRY RUN first before actual VACUUM!

</details>

## Solution: Task 7 (Run VACUUM)
<br/>
<details>

```sql
-- Run VACUUM (actually delete files)
VACUUM workspace.default.bookings_lab RETAIN 0 HOURS
```

**Expected output:**
* List of deleted file paths
* These files are permanently removed from storage
* Storage space is freed

**What happened:**
1. VACUUM found files older than 0 hours
2. Verified they're not in current table version
3. Permanently deleted them from cloud storage
4. Freed up disk space

**Important:**
* Time travel to old versions will now fail
* This is permanent - files cannot be recovered
* In production, use 7+ days retention

**Production example:**
```sql
-- Safe production VACUUM (7 days retention)
VACUUM workspace.default.bookings_lab RETAIN 168 HOURS
```

</details>

## Solution: Task 8 (Check Table History)
<br/>
<details>

```sql
DESCRIBE HISTORY workspace.default.bookings_lab
```

**What you should see:**

Multiple versions showing:
1. Version 0: `CREATE OR REPLACE TABLE`
2. Versions 1-4: `WRITE` (append operations)
3. Version 5: `OPTIMIZE`
4. Version 6: `WRITE` (batch 6)
5. Version 7: `OPTIMIZE` (with ZORDER)
6. Version 8: `VACUUM`

**Key insights:**
* Each operation creates a new version
* OPTIMIZE shows files added/removed
* Complete audit trail of all changes

</details>

## Solution: Task 9 (Final Table Health Check)
<br/>
<details>

**Check table details:**
```sql
DESCRIBE DETAIL workspace.default.bookings_lab
```

**Expected results:**
* `numFiles`: 1 (or very few)
* `sizeInBytes`: ~1-2 MB
* Much better than 5+ small files!

**Verify data integrity:**
```sql
SELECT COUNT(*) AS total_rows
FROM workspace.default.bookings_lab
```
Should be 7000 rows.

**Sample the data:**
```sql
SELECT * 
FROM workspace.default.bookings_lab
ORDER BY booking_id
LIMIT 20
```

**Success criteria:**
* Fewer files (1 vs 5+)
* All 7000 rows present
* Data intact and queryable
* Better performance for queries

**What we accomplished:**
1. Created table with small files (the problem)
2. Used OPTIMIZE to compact files (the solution)
3. Used ZORDER to cluster data (performance boost)
4. Used VACUUM to clean up old files (save storage)
5. Verified everything works (data integrity)

</details>

## Solution: Task 10 (Liquid Clustering)
<br/>
<details>

**Part A: Create a liquid-clustered table**
```sql
-- Create table with liquid clustering on region
CREATE OR REPLACE TABLE workspace.default.bookings_lc_lab
CLUSTER BY (region)
AS SELECT * FROM workspace.default.bookings_lab;

-- Verify clustering is configured
DESCRIBE DETAIL workspace.default.bookings_lc_lab;
```

**What you should see:**
* `clusteringColumns`: `["region"]`
* The table was created with data already partially clustered
* `numFiles`: Likely 1 (data was written in one pass)

**Part B: Change clustering keys**
```sql
-- Change to multi-column clustering (no data rewrite!)
ALTER TABLE workspace.default.bookings_lc_lab
CLUSTER BY (region, booking_date);

-- Trigger reclustering with new keys
OPTIMIZE workspace.default.bookings_lc_lab;

-- Verify the change
DESCRIBE DETAIL workspace.default.bookings_lc_lab;
```

**What you should see:**
* `clusteringColumns`: `["region", "booking_date"]`
* OPTIMIZE output shows reclustering metrics

**Optional: Remove clustering**
```sql
ALTER TABLE workspace.default.bookings_lc_lab
CLUSTER BY NONE;
```

**Key takeaways:**
* `CLUSTER BY` replaces ZORDER – simpler, incremental, and changeable
* Changing keys is instant (`ALTER TABLE`) – no data rewrite
* `OPTIMIZE` on a liquid-clustered table triggers reclustering (no `ZORDER BY` needed)
* `ZORDER BY` is rejected on a liquid-clustered table

</details>

## Solution: Task 11 (ANALYZE TABLE)
<br/>
<details>

**Part A: Table-level statistics**
```sql
-- Compute row count and size stats
ANALYZE TABLE workspace.default.bookings_lab COMPUTE STATISTICS;

-- Verify – look for the "Statistics" row
DESC EXTENDED workspace.default.bookings_lab;
```

**What you should see:**
* In `DESC EXTENDED` output, find the `Statistics` row
* It should show something like: `XXXX bytes, 7000 rows`

**Part B: Column-level statistics**
```sql
-- Compute stats for key columns
ANALYZE TABLE workspace.default.bookings_lab
COMPUTE STATISTICS FOR COLUMNS region, amount, booking_date;

-- View detailed stats for a column
DESC EXTENDED workspace.default.bookings_lab region;
```

**What you should see for `region`:**
* `distinct_count`: 4 (North, South, East, West)
* `num_nulls`: 0
* `avg_col_len` / `max_col_len`: string length stats

**What you should see for `amount`:**
* `min`: ~50
* `max`: ~1000
* `distinct_count`: high (many unique amounts)
* `num_nulls`: 0

**Part C: Delta-specific statistics**
```sql
ANALYZE TABLE workspace.default.bookings_lab COMPUTE DELTA STATISTICS;
```

This recomputes per-file min/max stats used for **data skipping**.

**Why this matters:**
* The optimizer uses these stats to pick the best join strategy
* Column stats help estimate selectivity of WHERE clauses
* Delta stats enable data skipping (files are skipped if their min/max don't match the filter)
* In production, **Predictive Optimization** runs this automatically for UC managed tables

</details>